## Assignment Tasks

```
1.	Exploratory Data Analysis (EDA)
o	Visualize trends, seasonality, and anomalies in the milk production data.
o	Check for any missing values or outliers.
o	Normalize or scale the data for neural network models.

2.	Data Preparation for Deep Learning
o	Create input-output sequences (time windows) suitable for training RNNs/LSTMs/GRUs.
o	Split data into training, validation, and test sets.
o	Reshape data for model input dimensions.

3.	Model Building
o	Build three separate models:
	Basic RNN
	LSTM
	GRU
o	Tune hyperparameters (e.g., window size, number of units, batch size, epochs).
o	Use appropriate loss functions and optimizers.

4.	Model Evaluation
o	Plot predictions vs. actual values.
o	Calculate forecasting metrics: RMSE, MAE, MAPE.
o	Compare the performance of RNN, LSTM, and GRU.

5.	Prediction and Visualization
o	Forecast milk production for the next 12 months.
o	Visualize the predicted trend with uncertainty or confidence intervals if possible.

6.	Business Insights
o	Interpret results and recommend how the dairy business can use these forecasts for better planning and resource allocation.

```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading the dataset into the enviroment
dataset = pd.read_csv("monthly_milk_production.csv")

# Working On copy set
df = dataset.copy()


#Understanding the data
print("\n<-------------INFO------------>\n")
print(df.info())

print("\n<-------------DESCRIBE ONLY NUMERICAL------------>\n")
print(df.describe())

print("\n<-------------DESCRIBE NUMERICAL AND CATEGORICAL------------>\n")
print(df.describe(include='all'))

print("\n<-------------MISSING VALUES------------>\n")
print(df.isnull().sum())

print("\n<-------------DUPLICATE VALUES--------->\n")
print(df["Date"].duplicated().sum())

In [ ]:
import matplotlib.dates as mdates

df['Date'] = df['Date'].astype('datetime64[ns]')


plt.figure(figsize=(15,8))
plt.gca().xaxis.set_major_locator(mdates.YearLocator(base=1))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


plt.plot(df['Date'],df['Production'])
plt.show()

## Preprocessing : Scaling and making sequence

In [ ]:
from sklearn.preprocessing import MinMaxScaler

series = df['Production'].astype('float')

scaler = MinMaxScaler()

series_scaled = scaler.fit_transform(series.values.reshape(-1,1))


# To create windows for our models
def make_sequence(data,size_window):
    X,y=[],[]
    for i in range(len(series) - size_window):
        X.append(data[i:i+size_window])
        y.append(data[i+size_window])
    X =np.array(X)
    y = np.array(y)
    return X.reshape(X.shape[0],X.shape[1],1),y

window = 12 # Taking  12 months consideration to predict the next mont
X,y =   make_sequence(series_scaled,window)

In [ ]:
# Train , test and validation split for RNN
n = len(X)
train_end = int(n*0.70)
val_end= int(n*0.85)

# 70 % the train data 15% are validation and test data
X_train,y_train = X[:train_end],y[:train_end]
X_val,y_val = X[train_end:val_end], y[train_end:val_end]
X_test,y_test = X[val_end:], y[val_end:]

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# Model Building
from tensorflow.keras import layers, models , callbacks

#using a simple function where we can pass all our models 
def model_builder(input_shape,units=64, cell='LSTM',dropout=0.2):
    model = models.Sequential()
    if cell == 'SimpleRNN':
        model.add(layers.SimpleRNN(units,input_shape=input_shape))
    elif cell == 'GRU':
        model.add(layers.GRU(units,input_shape=input_shape))
    else:
        model.add(layers.LSTM(units,input_shape=input_shape))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam',loss='mse')
    return model

In [ ]:
# Training models with early stoppings

es= callbacks.EarlyStopping(monitor ='val_loss',patience=10,restore_best_weights=True)


models_trained={}
histories={}
for cell in ['SimpleRNN','LSTM','GRU']:
    model = model_builder(input_shape=(window,1),units=64,cell= cell,dropout=0.2)
    build_model = model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=100,batch_size=32,callbacks=[es])
    models_trained[cell]=model
    histories[cell] = build_model
print(models_trained)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error


def evaluation(model,X_test,y_test):

    y_true = y_test.flatten()
        
    y_pred = model.predict(X_test)
    y_pred = y_pred.flatten()

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    return {"MSE": mse, "MAE":mae, "RMSE":rmse}, y_true,y_pred


In [ ]:

result={}
for cell,model in models_trained.items():
    res, y_true, y_pred = evaluation(model,X_test, y_test)
    result[cell] = res
for k,val in result.items():
    print(k, '----->',val)

In [ ]:
# Finding best model using low rmse value
best_model = min(result, key=lambda x: result[x]['RMSE'])

_,y_true, y_pred = evaluation(models_trained[best_model], X_test, y_test)

plt.figure(figsize=(15,10))
plt.plot(y_true,label="Actual")
plt.plot(y_pred,label="predicted")
plt.title("Actual vs Predicted")
plt.legend()
plt.show()

In [ ]:
# Forecasting next 12 months
def forecast_future(model, data, win, steps, scaler):
    # flatten input in case data has shape (n,1)
    history = list(data[-win:].flatten())
    preds = []
    for _ in range(steps):
        x = np.array(history[-win:], dtype=float).reshape(1, win, 1)
        y_predic = model.predict(x, verbose=0)[0, 0]
        history.append(y_predic)
        preds.append(y_predic)
    return scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()

# Forecast 12 months ahead
steps = 12 
future_index = pd.date_range(df.index[-1] + pd.offsets.MonthBegin(), periods=steps, freq='MS')

forecast = pd.Series(
    forecast_future(models_trained[best_model], series_scaled, window, steps, scaler),
    index=future_index,
    name="Forecast"
)

# Plotting actual vs forecast
plt.figure(figsize=(15, 10))
plt.plot(df['Production'].iloc[-36:], label="Actual")
plt.plot(forecast, label="Predicted", linestyle="--", marker="o")
plt.title("Actual vs Predicted (12-Month Forecast)")
plt.legend()
plt.show()


## Recommending and interpreting:
```
1.Labor & Time Off: Hire seasonal help early (by February) for the peak. Schedule all major maintenance and core staff leave during the low-production winter months.

2.Supply Chain: Secure extra trucks and immediate buyers now for the high season's surplus. During the winter low, cut back on temporary distribution costs.

3.Inventory & Sales: Convert excess milk (nearly 140 unit swing) into higher-value, storable goods (like cheese) to maximize profits and manage tank capacity.

4.Costs & Growth: Buy your bulk feed in the summer to lock in lower prices for the winter. Plan capital investments (like new equipment) to match the slight upward trend in production predicted for the future.
```
